In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.model_selection import LeaveOneGroupOut
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from sklearn.ensemble import ExtraTreesRegressor, RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.linear_model import Ridge, ElasticNet
from sklearn.svm import SVR

# ---------------------------------------------------------
# 0) 데이터 준비
# ---------------------------------------------------------
CSV_PATH = r"C:\Users\공주님\Documents\배터리_대시보드\with_soh_rul_.csv"

discharge1 = pd.read_csv(CSV_PATH)
df0 = discharge1.copy()

# (선택) 특정 배터리만 먼저
target_batts = ["B0005", "B0006", "B0007", "B0018"]
df0 = df0[df0["battery_id"].isin(target_batts)].copy()

# 타깃 선택
TARGET = "SOH"   # 또는 "RUL"
CYCLE_COL = "cycle_id"

# 사용할 기본 피처들 (너가 선정한 코어)
BASE_FEATURES = [
    "discharge_t_to_V_below_3.5",
    "discharge_V_mean",
    "slope_Vmeas_50_1500",
    "HI2_Max_Temp",
    "discharge_E_Wh_abs",
    'discharge_Temp_std'
]

# 0-1) 숫자 변환
for c in [TARGET, CYCLE_COL] + BASE_FEATURES:
    if c in df0.columns:
        df0[c] = pd.to_numeric(df0[c], errors="coerce")

# (중요) RUL은 0 이상만 쓰고 싶다면
if TARGET == "RUL" and "RUL" in df0.columns:
    df0 = df0[df0["RUL"] >= 0].copy()

# 0-2) 정렬(시계열 안전성)
df0 = df0.sort_values(["battery_id", CYCLE_COL]).reset_index(drop=True)

# ---------------------------------------------------------
# 1) 시계열 피처 생성 (손실 최소: lag1 + diff1 + rolling mean(3))
# ---------------------------------------------------------
LAGS = [1]          # 필요하면 [1,2]로
ROLL_W = 5          # 필요하면 5로

df_ts = df0.copy()

# lag / diff / rolling
for c in BASE_FEATURES:
    # lag
    for k in LAGS:
        df_ts[f"{c}_lag{k}"] = df_ts.groupby("battery_id")[c].shift(k)
    # diff (1차 변화량)
    df_ts[f"{c}_diff1"] = df_ts.groupby("battery_id")[c].diff(1)
    # rolling mean (짧게)
    df_ts[f"{c}_rm{ROLL_W}"] = (
        df_ts.groupby("battery_id")[c]
             .rolling(ROLL_W)
             .mean()
             .reset_index(level=0, drop=True)
    )

# 최종 FEATURES 구성
LAG_FEATURES   = [f"{c}_lag{k}" for c in BASE_FEATURES for k in LAGS]
DIFF_FEATURES  = [f"{c}_diff1" for c in BASE_FEATURES]
ROLL_FEATURES  = [f"{c}_rm{ROLL_W}" for c in BASE_FEATURES]

FEATURES = BASE_FEATURES + LAG_FEATURES + DIFF_FEATURES + ROLL_FEATURES

use_cols = ["battery_id", CYCLE_COL, TARGET] + FEATURES
df = df_ts[use_cols].copy()

# 숫자 변환 다시 한 번 (생성 컬럼 포함)
for c in [TARGET] + FEATURES:
    df[c] = pd.to_numeric(df[c], errors="coerce")

# 결측 제거 (lag/rolling 때문에 앞쪽 몇 행은 NaN)
df = df.dropna().reset_index(drop=True)

print("data shape (after ts features):", df.shape)
display(df.head())

# ---------------------------------------------------------
# 2) LOBO 설정
# ---------------------------------------------------------
X = df[FEATURES]
y = df[TARGET].values
groups = df["battery_id"].values

logo = LeaveOneGroupOut()
numeric_features = FEATURES

# 전처리: 결측 -> 중앙값 대치 + 스케일링(선형/커널에 유리)
preprocess_scaled = ColumnTransformer(
    transformers=[
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]), numeric_features)
    ],
    remainder="drop"
)

# 트리 계열은 스케일링 필수 아님
preprocess_noscale = ColumnTransformer(
    transformers=[
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
        ]), numeric_features)
    ],
    remainder="drop"
)

# ---------------------------------------------------------
# 3) 비교할 모델들 (원래 + XGB/Cat)
# ---------------------------------------------------------
try:
    from xgboost import XGBRegressor
except Exception as e:
    XGBRegressor = None
    print("⚠️ xgboost import 실패:", e)

try:
    from catboost import CatBoostRegressor
except Exception as e:
    CatBoostRegressor = None
    print("⚠️ catboost import 실패:", e)

models = {
    "ExtraTrees": Pipeline([
        ("prep", preprocess_noscale),
        ("model", ExtraTreesRegressor(
            n_estimators=800,
            random_state=42,
            n_jobs=-1
        ))
    ]),
    "RandomForest": Pipeline([
        ("prep", preprocess_noscale),
        ("model", RandomForestRegressor(
            n_estimators=800,
            random_state=42,
            n_jobs=-1
        ))
    ]),
    "HistGB": Pipeline([
        ("prep", preprocess_noscale),
        ("model", HistGradientBoostingRegressor(
            random_state=42
        ))
    ]),
    "Ridge": Pipeline([
        ("prep", preprocess_scaled),
        ("model", Ridge(alpha=1.0))
    ]),
    "ElasticNet": Pipeline([
        ("prep", preprocess_scaled),
        ("model", ElasticNet(random_state=42))
    ]),
    "SVR(RBF)": Pipeline([
        ("prep", preprocess_scaled),
        ("model", SVR(kernel="rbf"))
    ]),
}

if XGBRegressor is not None:
    models["XGBoost"] = Pipeline([
        ("prep", preprocess_noscale),
        ("model", XGBRegressor(
            n_estimators=1000,
            learning_rate=0.05,
            max_depth=6,
            subsample=0.9,
            colsample_bytree=0.9,
            reg_lambda=1.0,
            random_state=42,
            n_jobs=-1,
            objective="reg:squarederror"
        ))
    ])

if CatBoostRegressor is not None:
    models["CatBoost"] = Pipeline([
        ("prep", preprocess_noscale),
        ("model", CatBoostRegressor(
            iterations=3000,
            learning_rate=0.03,
            depth=6,
            loss_function="MAE",
            random_seed=42,
            verbose=False
        ))
    ])

# ---------------------------------------------------------
# 4) LOBO 평가 함수
# ---------------------------------------------------------
def lobo_evaluate(model, X, y, groups, logo):
    fold_rows = []
    for train_idx, test_idx in logo.split(X, y, groups=groups):
        X_tr, X_te = X.iloc[train_idx], X.iloc[test_idx]
        y_tr, y_te = y[train_idx], y[test_idx]
        batt = groups[test_idx][0]

        model.fit(X_tr, y_tr)
        pred = model.predict(X_te)

        mae = mean_absolute_error(y_te, pred)
        rmse = mean_squared_error(y_te, pred) ** 0.5
        r2 = r2_score(y_te, pred)

        fold_rows.append([batt, len(test_idx), mae, rmse, r2])

    fold_df = pd.DataFrame(fold_rows, columns=["test_battery", "n_test", "MAE", "RMSE", "R2"])
    summary = {
        "MAE_mean": fold_df["MAE"].mean(),
        "MAE_std": fold_df["MAE"].std(),
        "RMSE_mean": fold_df["RMSE"].mean(),
        "RMSE_std": fold_df["RMSE"].std(),
        "R2_mean": fold_df["R2"].mean(),
        "R2_std": fold_df["R2"].std(),
        "n_folds": len(fold_df),
        "n_samples": len(X),
        "n_batteries": len(np.unique(groups)),
    }
    return summary, fold_df

# ---------------------------------------------------------
# 5) 모델 전체 “딸깍 비교”
# ---------------------------------------------------------
all_summaries = []
all_folds = {}

for name, model in models.items():
    summary, fold_df = lobo_evaluate(model, X, y, groups, logo)
    all_summaries.append([
        name,
        summary["MAE_mean"], summary["RMSE_mean"], summary["R2_mean"],
        summary["MAE_std"], summary["RMSE_std"], summary["R2_std"],
        summary["n_folds"], summary["n_samples"], summary["n_batteries"]
    ])
    all_folds[name] = fold_df

result = pd.DataFrame(all_summaries, columns=[
    "model", "MAE", "RMSE", "R2",
    "MAE_std", "RMSE_std", "R2_std",
    "n_folds", "n_samples", "n_batteries"
]).sort_values("MAE")

print("=" * 90)
print(f"[LOBO AutoML + TS Features] Target = {TARGET} | Batteries = {sorted(df['battery_id'].unique())}")
print(f"Base feats: {len(BASE_FEATURES)} | +lags:{len(LAG_FEATURES)} +diff:{len(DIFF_FEATURES)} +roll:{len(ROLL_FEATURES)} => total {len(FEATURES)}")
print("=" * 90)
display(result)

# ---------------------------------------------------------
# 6) 베스트 모델 배터리별 성능
# ---------------------------------------------------------
best_name = result.iloc[0]["model"]
print("\nBest model:", best_name)
display(all_folds[best_name].sort_values("MAE"))

data shape (after ts features): (620, 27)


,battery_id,cycle_id,SOH,discharge_t_to_V_below_3.5,discharge_V_mean,slope_Vmeas_50_1500,HI2_Max_Temp,discharge_E_Wh_abs,discharge_Temp_std,discharge_t_to_V_below_3.5_lag1,...,slope_Vmeas_50_1500_diff1,HI2_Max_Temp_diff1,discharge_E_Wh_abs_diff1,discharge_Temp_std_diff1,discharge_t_to_V_below_3.5_rm5,discharge_V_mean_rm5,slope_Vmeas_50_1500_rm5,HI2_Max_Temp_rm5,discharge_E_Wh_abs_rm5,discharge_Temp_std_rm5
0,B0005,5.0,0.988235,2121.794596,3.544875,-0.000238,38.665393,6.561344,3.404667,2117.469577,...,-5.075432e-07,-0.096912,-0.002195,0.009361,2097.847822,3.541710,-0.000236,38.852415,6.580198,3.423892
1,B0005,6.0,0.988782,2133.519013,3.544142,-0.000239,38.751695,6.566809,3.426122,2121.794596,...,-5.372219e-07,0.086302,0.005465,0.021455,2115.314294,3.544142,-0.000237,38.806317,6.570111,3.409956
2,B0005,7.0,0.988504,2142.412264,3.543990,-0.000239,38.820701,6.566596,3.428344,2133.519013,...,-5.629843e-07,0.069006,-0.000212,0.002223,2124.808686,3.544994,-0.000238,38.763778,6.564423,3.408523
3,B0005,8.0,0.983447,2149.201807,3.555909,-0.000242,38.517130,6.543075,3.360425,2142.412264,...,-2.264793e-06,-0.303571,-0.023521,-0.067920,2132.879451,3.546984,-0.000239,38.703445,6.560273,3.402973
4,B0005,9.0,0.982917,2146.916797,3.554758,-0.000241,38.526268,6.537439,3.381314,2149.201807,...,7.762757e-07,0.009138,-0.005637,0.020890,2138.768895,3.548735,-0.000240,38.656237,6.555053,3.400174


[LOBO AutoML + TS Features] Target = SOH | Batteries = ['B0005', 'B0006', 'B0007', 'B0018']
Base feats: 6 | +lags:6 +diff:6 +roll:6 => total 24


,model,MAE,RMSE,R2,MAE_std,RMSE_std,R2_std,n_folds,n_samples,n_batteries
3,Ridge,0.016280,0.019441,0.956320,0.009252,0.011679,0.035695,4,620,4
7,CatBoost,0.022631,0.027335,0.906904,0.019110,0.021145,0.101105,4,620,4
0,ExtraTrees,0.026120,0.029497,0.895057,0.018774,0.020181,0.099555,4,620,4
2,HistGB,0.026315,0.029695,0.889640,0.021130,0.022634,0.117016,4,620,4
6,XGBoost,0.027269,0.030107,0.891384,0.019369,0.020890,0.105912,4,620,4
1,RandomForest,0.028012,0.030734,0.886502,0.020025,0.021516,0.111768,4,620,4
5,SVR(RBF),0.057484,0.071134,0.430099,0.035323,0.039729,0.429860,4,620,4
4,ElasticNet,0.093852,0.109809,-0.300881,0.027358,0.030353,0.299666,4,620,4



Best model: Ridge


,test_battery,n_test,MAE,RMSE,R2
2,B0007,164,0.007287,0.008944,0.988536
0,B0005,164,0.012481,0.014834,0.978383
3,B0018,128,0.016364,0.017947,0.949489
1,B0006,164,0.028988,0.036040,0.908874


In [2]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.model_selection import LeaveOneGroupOut
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


# =========================================================
# 0) 설정 (너 경로 그대로)
# =========================================================
CSV_PATH  = r"C:\Users\공주님\Documents\8282_streamlit\조원_with_soh_rul_25일버전.csv"
TARGET    = "SOH"
GROUP_COL = "battery_id"
CYCLE_COL = "cycle_id"

target_batts = ["B0005", "B0006", "B0007", "B0018"]

# ✅ 현재 너의 베스트 코어(5개)
BASE_CORE = [
    "discharge_t_to_V_below_3.5",
    "discharge_V_mean",
    "slope_Vmeas_50_1500",
    "HI2_Max_Temp",
    "discharge_E_Wh_abs",
]

# ✅ Ridge 친화 후보 3개 (있을 때만 실험)
CANDIDATE_3 = [
    "discharge_Temp_std",
    "discharge_t_to_Tmax",
    "discharge_Decrement_3.4_3.2_s",
]

# 시계열 피처 생성 세팅
LAGS   = [1]
ROLL_W = 5


# =========================================================
# 1) 공통 유틸: TS 피처 생성 + 정리
# =========================================================
def build_ts_table(df0: pd.DataFrame, base_features: list) -> tuple[pd.DataFrame, list]:
    """
    base_features를 기준으로
    - lag1, diff1, rm(ROLL_W) 생성
    - dropna
    반환: (df_final, FEATURES_ALL)
    """
    df = df0.copy()
    df = df.sort_values([GROUP_COL, CYCLE_COL]).reset_index(drop=True)

    # 숫자화
    for c in [TARGET, CYCLE_COL] + base_features:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")

    df_ts = df.copy()

    for c in base_features:
        for k in LAGS:
            df_ts[f"{c}_lag{k}"] = df_ts.groupby(GROUP_COL)[c].shift(k)
        df_ts[f"{c}_diff1"] = df_ts.groupby(GROUP_COL)[c].diff(1)
        df_ts[f"{c}_rm{ROLL_W}"] = (
            df_ts.groupby(GROUP_COL)[c].rolling(ROLL_W).mean().reset_index(level=0, drop=True)
        )

    lag_feats  = [f"{c}_lag{k}" for c in base_features for k in LAGS]
    diff_feats = [f"{c}_diff1" for c in base_features]
    roll_feats = [f"{c}_rm{ROLL_W}" for c in base_features]

    features_all = base_features + lag_feats + diff_feats + roll_feats

    use_cols = [GROUP_COL, CYCLE_COL, TARGET] + features_all
    df_out = df_ts[use_cols].copy()

    # 다시 숫자화 (생성 컬럼 포함)
    for c in [TARGET] + features_all:
        df_out[c] = pd.to_numeric(df_out[c], errors="coerce")

    df_out = df_out.dropna().reset_index(drop=True)
    return df_out, features_all


def lobo_ridge_eval(df: pd.DataFrame, features: list, alpha: float) -> tuple[dict, pd.DataFrame]:
    """
    LOBO로 Ridge(alpha) 평가
    반환: (summary_dict, fold_df)
    """
    X = df[features]
    y = df[TARGET].values
    groups = df[GROUP_COL].values

    logo = LeaveOneGroupOut()

    preprocess = ColumnTransformer(
        transformers=[("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]), features)],
        remainder="drop"
    )

    model = Pipeline([
        ("prep", preprocess),
        ("model", Ridge(alpha=alpha))
    ])

    rows = []
    for tr, te in logo.split(X, y, groups=groups):
        batt = groups[te][0]
        model.fit(X.iloc[tr], y[tr])
        pred = model.predict(X.iloc[te])

        rows.append([
            batt, len(te),
            mean_absolute_error(y[te], pred),
            mean_squared_error(y[te], pred) ** 0.5,
            r2_score(y[te], pred)
        ])

    fold_df = pd.DataFrame(rows, columns=["test_battery", "n_test", "MAE", "RMSE", "R2"])
    summary = {
        "alpha": alpha,
        "MAE": fold_df["MAE"].mean(),
        "RMSE": fold_df["RMSE"].mean(),
        "R2": fold_df["R2"].mean(),
        "MAE_std": fold_df["MAE"].std(),
        "RMSE_std": fold_df["RMSE"].std(),
        "R2_std": fold_df["R2"].std(),
        "n_samples": len(df),
        "n_features": len(features),
    }
    return summary, fold_df


# =========================================================
# 2) 데이터 로드 + 필터
# =========================================================
df0 = pd.read_csv(CSV_PATH)
df0 = df0[df0[GROUP_COL].isin(target_batts)].copy()

# cycle/target 존재 확인
assert CYCLE_COL in df0.columns, f"'{CYCLE_COL}' 컬럼이 없음"
assert TARGET in df0.columns, f"'{TARGET}' 컬럼이 없음"


# =========================================================
# (1) alpha 튜닝 LOBO 표 (코어 5개 + TS)
# =========================================================
df_base, FEATURES_BASE_ALL = build_ts_table(df0, BASE_CORE)

alphas = [0.001, 0.003, 0.01, 0.03, 0.1, 0.3, 1, 3, 10, 30, 100]
alpha_rows = []
alpha_folds = {}

for a in alphas:
    summary, fold_df = lobo_ridge_eval(df_base, FEATURES_BASE_ALL, alpha=a)
    alpha_rows.append(summary)
    alpha_folds[a] = fold_df

alpha_table = pd.DataFrame(alpha_rows).sort_values("MAE").reset_index(drop=True)

print("\n" + "="*95)
print("[(1) Ridge alpha 튜닝 | LOBO | BASE_CORE + TS(lag1+diff1+rm5)]")
print(f"BASE_CORE={BASE_CORE}")
print(f"Total features={len(FEATURES_BASE_ALL)} | Samples={len(df_base)}")
print("="*95)
display(alpha_table)

best_alpha = float(alpha_table.iloc[0]["alpha"])
print("\nBest alpha =", best_alpha)
display(alpha_folds[best_alpha].sort_values("MAE"))


# =========================================================
# (2) 후보 3개 추가 실험 표 (best_alpha 고정)
#     - BASE_CORE만
#     - BASE_CORE + 후보1
#     - BASE_CORE + 후보2
#     - BASE_CORE + 후보3
# =========================================================
def candidate_exists(col):
    return col in df0.columns

tests = [("BASE_ONLY", BASE_CORE)]
for c in CANDIDATE_3:
    if candidate_exists(c):
        tests.append((f"BASE+{c}", BASE_CORE + [c]))
    else:
        print(f"⚠️ 후보 컬럼 없음, skip: {c}")

cand_rows = []
cand_folds = {}

for name, feats in tests:
    df_tmp, feats_all = build_ts_table(df0, feats)
    summary, fold_df = lobo_ridge_eval(df_tmp, feats_all, alpha=best_alpha)

    summary_out = {
        "set": name,
        "MAE": summary["MAE"],
        "RMSE": summary["RMSE"],
        "R2": summary["R2"],
        "MAE_std": summary["MAE_std"],
        "RMSE_std": summary["RMSE_std"],
        "R2_std": summary["R2_std"],
        "n_samples": summary["n_samples"],
        "n_features": summary["n_features"],
        "base_features": len(feats),
    }
    cand_rows.append(summary_out)
    cand_folds[name] = fold_df

cand_table = pd.DataFrame(cand_rows).sort_values("MAE").reset_index(drop=True)

print("\n" + "="*95)
print("[(2) 후보 3개 추가 실험 | LOBO | best_alpha 고정]")
print(f"best_alpha={best_alpha}")
print("="*95)
display(cand_table)

best_set = cand_table.iloc[0]["set"]
print("\nBest feature set =", best_set)
display(cand_folds[best_set].sort_values("MAE"))


[(1) Ridge alpha 튜닝 | LOBO | BASE_CORE + TS(lag1+diff1+rm5)]
BASE_CORE=['discharge_t_to_V_below_3.5', 'discharge_V_mean', 'slope_Vmeas_50_1500', 'HI2_Max_Temp', 'discharge_E_Wh_abs']
Total features=20 | Samples=620


,alpha,MAE,RMSE,R2,MAE_std,RMSE_std,R2_std,n_samples,n_features
0,3.000,0.015300,0.018587,0.954826,0.010708,0.012934,0.042126,620,20
1,1.000,0.015794,0.019132,0.957267,0.009030,0.011726,0.035214,620,20
2,0.300,0.016565,0.019866,0.956233,0.008342,0.011146,0.032717,620,20
3,0.100,0.016814,0.020151,0.955527,0.008169,0.010959,0.031911,620,20
4,0.030,0.016935,0.020314,0.954999,0.008003,0.010814,0.031339,620,20
5,10.000,0.017014,0.020061,0.945093,0.011002,0.012876,0.053556,620,20
6,0.010,0.017030,0.020441,0.954502,0.007870,0.010706,0.030961,620,20
7,0.003,0.017098,0.020533,0.954116,0.007781,0.010634,0.030729,620,20
8,0.001,0.017126,0.020570,0.953955,0.007746,0.010605,0.030641,620,20
9,100.000,0.018023,0.021414,0.943878,0.007809,0.009336,0.041423,620,20



Best alpha = 3.0


,test_battery,n_test,MAE,RMSE,R2
2,B0007,164,0.003621,0.005312,0.995956
0,B0005,164,0.009373,0.011918,0.986048
3,B0018,128,0.021148,0.022142,0.923116
1,B0006,164,0.027058,0.034975,0.914182



[(2) 후보 3개 추가 실험 | LOBO | best_alpha 고정]
best_alpha=3.0


,set,MAE,RMSE,R2,MAE_std,RMSE_std,R2_std,n_samples,n_features,base_features
0,BASE_ONLY,0.015300,0.018587,0.954826,0.010708,0.012934,0.042126,620,20,5
1,BASE+discharge_Decrement_3.4_3.2_s,0.015440,0.018499,0.957536,0.008407,0.010211,0.034665,620,24,6
2,BASE+discharge_t_to_Tmax,0.015803,0.018725,0.954125,0.010146,0.012651,0.042830,620,24,6
3,BASE+discharge_Temp_std,0.015815,0.018904,0.954210,0.010920,0.012955,0.042249,620,24,6



Best feature set = BASE_ONLY


,test_battery,n_test,MAE,RMSE,R2
2,B0007,164,0.003621,0.005312,0.995956
0,B0005,164,0.009373,0.011918,0.986048
3,B0018,128,0.021148,0.022142,0.923116
1,B0006,164,0.027058,0.034975,0.914182


결론적으로 지금 alpha = 3일때 가장 MAE 수치가 낮다.

Z_SCORE 적용

In [3]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.model_selection import LeaveOneGroupOut
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


# =========================================================
# 0) 설정
# =========================================================
CSV_PATH  = r"C:\Users\공주님\Documents\8282_streamlit\조원_with_soh_rul_25일버전.csv"
TARGET    = "SOH"
GROUP_COL = "battery_id"
CYCLE_COL = "cycle_id"

target_batts = ["B0005", "B0006", "B0007", "B0018"]

# ✅ 베스트 코어 5개
BASE_CORE = [
    "discharge_t_to_V_below_3.5",
    "discharge_V_mean",
    "slope_Vmeas_50_1500",
    "HI2_Max_Temp",
    "discharge_E_Wh_abs",
]

# TS feature 세팅
LAGS   = [1]
ROLL_W = 5

# Ridge alpha (너 결론)
ALPHA = 3.0


# =========================================================
# 1) 유틸: TS feature 생성
# =========================================================
def build_ts_table(df0: pd.DataFrame, base_features: list):
    df = df0.copy()
    df = df.sort_values([GROUP_COL, CYCLE_COL]).reset_index(drop=True)

    # 숫자화
    for c in [TARGET, CYCLE_COL] + base_features:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    df_ts = df.copy()

    for c in base_features:
        for k in LAGS:
            df_ts[f"{c}_lag{k}"] = df_ts.groupby(GROUP_COL)[c].shift(k)
        df_ts[f"{c}_diff1"] = df_ts.groupby(GROUP_COL)[c].diff(1)
        df_ts[f"{c}_rm{ROLL_W}"] = (
            df_ts.groupby(GROUP_COL)[c].rolling(ROLL_W).mean().reset_index(level=0, drop=True)
        )

    lag_feats  = [f"{c}_lag{k}" for c in base_features for k in LAGS]
    diff_feats = [f"{c}_diff1" for c in base_features]
    roll_feats = [f"{c}_rm{ROLL_W}" for c in base_features]

    features_all = base_features + lag_feats + diff_feats + roll_feats

    use_cols = [GROUP_COL, CYCLE_COL, TARGET] + features_all
    df_out = df_ts[use_cols].copy()

    # 숫자화(생성 컬럼 포함)
    for c in [TARGET] + features_all:
        df_out[c] = pd.to_numeric(df_out[c], errors="coerce")

    df_out = df_out.dropna().reset_index(drop=True)
    return df_out, features_all


# =========================================================
# 2) 배터리별 z-score (2버전)
# =========================================================
def batt_zscore_full(df: pd.DataFrame, features: list):
    """
    (버전1) 배터리 전체 구간 mean/std로 z-score (쉽지만 분포 통계를 미래까지 봄)
    """
    out = df.copy()
    g = out.groupby(GROUP_COL)
    for c in features:
        mu = g[c].transform("mean")
        sd = g[c].transform("std")
        sd = sd.replace(0, np.nan)
        out[c] = (out[c] - mu) / sd
    # std=0이거나 NaN이면 0으로
    out[features] = out[features].fillna(0.0)
    return out


def batt_zscore_expanding(df: pd.DataFrame, features: list):
    """
    (버전2) expanding(과거만) mean/std로 z-score (시계열 누수 최소)
    - cycle t의 스케일링에 1..t까지 사용
    """
    out = df.copy()
    out = out.sort_values([GROUP_COL, CYCLE_COL]).reset_index(drop=True)

    for c in features:
        # expanding mean/std per battery
        grp = out.groupby(GROUP_COL)[c]
        exp_mean = grp.expanding().mean().reset_index(level=0, drop=True)
        exp_std  = grp.expanding().std().reset_index(level=0, drop=True)

        exp_std = exp_std.replace(0, np.nan)
        out[c] = (out[c] - exp_mean) / exp_std

    out[features] = out[features].fillna(0.0)
    return out


# =========================================================
# 3) LOBO 평가(Ridge)
# =========================================================
def lobo_ridge_eval(df: pd.DataFrame, features: list, alpha: float):
    X = df[features]
    y = df[TARGET].values
    groups = df[GROUP_COL].values

    logo = LeaveOneGroupOut()

    # 여기서는 이미 배터리별 z-score를 외부에서 했을 수 있으니
    # scaler는 "끄는 게" 일반적으로 맞음. (중복 스케일링 방지)
    preprocess = ColumnTransformer(
        transformers=[("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            # ("scaler", StandardScaler()),  # ❌ batt z-score면 보통 끔
        ]), features)],
        remainder="drop"
    )

    model = Pipeline([
        ("prep", preprocess),
        ("model", Ridge(alpha=alpha))
    ])

    rows = []
    for tr, te in logo.split(X, y, groups=groups):
        batt = groups[te][0]
        model.fit(X.iloc[tr], y[tr])
        pred = model.predict(X.iloc[te])
        rows.append([
            batt, len(te),
            mean_absolute_error(y[te], pred),
            mean_squared_error(y[te], pred) ** 0.5,
            r2_score(y[te], pred)
        ])

    fold = pd.DataFrame(rows, columns=["test_battery","n_test","MAE","RMSE","R2"])
    summary = {
        "MAE": fold["MAE"].mean(),
        "RMSE": fold["RMSE"].mean(),
        "R2": fold["R2"].mean(),
        "MAE_std": fold["MAE"].std(),
        "RMSE_std": fold["RMSE"].std(),
        "R2_std": fold["R2"].std(),
        "n_samples": len(df),
        "n_features": len(features),
    }
    return summary, fold


# =========================================================
# 4) 실행: baseline vs batt-z (full) vs batt-z (expanding)
# =========================================================
df0 = pd.read_csv(CSV_PATH)
df0 = df0[df0[GROUP_COL].isin(target_batts)].copy()

assert CYCLE_COL in df0.columns, f"'{CYCLE_COL}' 없음"
assert TARGET in df0.columns, f"'{TARGET}' 없음"

# TS 테이블 생성
df_base, FEATURES_ALL = build_ts_table(df0, BASE_CORE)

print("Base feats:", len(BASE_CORE), "| total features:", len(FEATURES_ALL), "| samples:", len(df_base))

# (A) baseline (배터리 z-score 안 함)
sum_base, fold_base = lobo_ridge_eval(df_base, FEATURES_ALL, alpha=ALPHA)

# (B) 배터리 전체구간 z-score
df_fullz = batt_zscore_full(df_base, FEATURES_ALL)
sum_fullz, fold_fullz = lobo_ridge_eval(df_fullz, FEATURES_ALL, alpha=ALPHA)

# (C) 과거만(expanding) z-score
df_expz = batt_zscore_expanding(df_base, FEATURES_ALL)
sum_expz, fold_expz = lobo_ridge_eval(df_expz, FEATURES_ALL, alpha=ALPHA)

# 결과 표
res = pd.DataFrame([
    ["baseline(no batt-z)", sum_base["MAE"], sum_base["RMSE"], sum_base["R2"], sum_base["MAE_std"], sum_base["n_samples"]],
    ["batt-z FULL(mean/std)", sum_fullz["MAE"], sum_fullz["RMSE"], sum_fullz["R2"], sum_fullz["MAE_std"], sum_fullz["n_samples"]],
    ["batt-z EXPANDING(past only)", sum_expz["MAE"], sum_expz["RMSE"], sum_expz["R2"], sum_expz["MAE_std"], sum_expz["n_samples"]],
], columns=["setting","MAE","RMSE","R2","MAE_std","n_samples"]).sort_values("MAE")

print("\n" + "="*95)
print(f"[LOBO Ridge(alpha={ALPHA})] BASE_CORE + lag1 + diff1 + rm{ROLL_W}")
print("="*95)
display(res)

print("\n--- Per-battery folds (baseline) ---")
display(fold_base.sort_values("MAE"))

print("\n--- Per-battery folds (batt-z FULL) ---")
display(fold_fullz.sort_values("MAE"))

print("\n--- Per-battery folds (batt-z EXPANDING) ---")
display(fold_expz.sort_values("MAE"))

Base feats: 5 | total features: 20 | samples: 620

[LOBO Ridge(alpha=3.0)] BASE_CORE + lag1 + diff1 + rm5


,setting,MAE,RMSE,R2,MAE_std,n_samples
0,baseline(no batt-z),0.013596,0.016261,0.966464,0.006908,620
1,batt-z FULL(mean/std),0.049881,0.053057,0.651973,0.034185,620
2,batt-z EXPANDING(past only),0.071539,0.088564,0.158495,0.027447,620



--- Per-battery folds (baseline) ---


,test_battery,n_test,MAE,RMSE,R2
2,B0007,164,0.006131,0.007378,0.992200
0,B0005,164,0.009396,0.010798,0.988546
3,B0018,128,0.018491,0.020578,0.933596
1,B0006,164,0.020366,0.026289,0.951515



--- Per-battery folds (batt-z FULL) ---


,test_battery,n_test,MAE,RMSE,R2
3,B0018,128,0.020796,0.024864,0.903045
0,B0005,164,0.026183,0.026986,0.928461
2,B0007,164,0.057491,0.060257,0.479640
1,B0006,164,0.095052,0.100120,0.296747



--- Per-battery folds (batt-z EXPANDING) ---


,test_battery,n_test,MAE,RMSE,R2
3,B0018,128,0.041129,0.053421,0.552462
2,B0007,164,0.066995,0.079938,0.084219
0,B0005,164,0.070258,0.086246,0.269305
1,B0006,164,0.107773,0.134651,-0.272007


Z_SCORE 적용? 지금 성능 다떨어진다. 하면 안됨 ㅎㅎ

In [4]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.model_selection import LeaveOneGroupOut
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from sklearn.ensemble import ExtraTreesRegressor, RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.linear_model import Ridge, ElasticNet
from sklearn.svm import SVR

# ---------------------------------------------------------
# 0) 데이터 준비
# ---------------------------------------------------------
CSV_PATH = r"C:\Users\공주님\Documents\8282_streamlit\조원_with_soh_rul_25일버전.csv"

discharge1 = pd.read_csv(CSV_PATH)
df0 = discharge1.copy()

# (선택) 특정 배터리만 먼저
target_batts = ["B0005", "B0006", "B0007", "B0018"]
df0 = df0[df0["battery_id"].isin(target_batts)].copy()

# 타깃 선택
TARGET = "SOH"   # 또는 "RUL"
CYCLE_COL = "cycle_id"

# 사용할 기본 피처들 (너가 선정한 코어)
BASE_FEATURES = [
    "discharge_t_to_V_below_3.5",
    "discharge_V_mean",
    "slope_Vmeas_50_1500",
    "HI2_Max_Temp",
    "discharge_E_Wh_abs"
]

# 0-1) 숫자 변환
for c in [TARGET, CYCLE_COL] + BASE_FEATURES:
    if c in df0.columns:
        df0[c] = pd.to_numeric(df0[c], errors="coerce")

# (중요) RUL은 0 이상만 쓰고 싶다면
if TARGET == "RUL" and "RUL" in df0.columns:
    df0 = df0[df0["RUL"] >= 0].copy()

# 0-2) 정렬(시계열 안전성)
df0 = df0.sort_values(["battery_id", CYCLE_COL]).reset_index(drop=True)

# ---------------------------------------------------------
# 1) 시계열 피처 생성 (손실 최소: lag1 + diff1 + rolling mean(3))
# ---------------------------------------------------------
LAGS = [1]          # 필요하면 [1,2]로
ROLL_W = 5          # 필요하면 5로

df_ts = df0.copy()

# lag / diff / rolling
for c in BASE_FEATURES:
    # lag
    for k in LAGS:
        df_ts[f"{c}_lag{k}"] = df_ts.groupby("battery_id")[c].shift(k)
    # diff (1차 변화량)
    df_ts[f"{c}_diff1"] = df_ts.groupby("battery_id")[c].diff(1)
    # rolling mean (짧게)
    df_ts[f"{c}_rm{ROLL_W}"] = (
        df_ts.groupby("battery_id")[c]
             .rolling(ROLL_W)
             .mean()
             .reset_index(level=0, drop=True)
    )

# 최종 FEATURES 구성
LAG_FEATURES   = [f"{c}_lag{k}" for c in BASE_FEATURES for k in LAGS]
DIFF_FEATURES  = [f"{c}_diff1" for c in BASE_FEATURES]
ROLL_FEATURES  = [f"{c}_rm{ROLL_W}" for c in BASE_FEATURES]

FEATURES = BASE_FEATURES + LAG_FEATURES + DIFF_FEATURES + ROLL_FEATURES

use_cols = ["battery_id", CYCLE_COL, TARGET] + FEATURES
df = df_ts[use_cols].copy()

# 숫자 변환 다시 한 번 (생성 컬럼 포함)
for c in [TARGET] + FEATURES:
    df[c] = pd.to_numeric(df[c], errors="coerce")

# 결측 제거 (lag/rolling 때문에 앞쪽 몇 행은 NaN)
df = df.dropna().reset_index(drop=True)

print("data shape (after ts features):", df.shape)
display(df.head())

# ---------------------------------------------------------
# 2) LOBO 설정
# ---------------------------------------------------------
X = df[FEATURES]
y = df[TARGET].values
groups = df["battery_id"].values

logo = LeaveOneGroupOut()
numeric_features = FEATURES

# 전처리: 결측 -> 중앙값 대치 + 스케일링(선형/커널에 유리)
preprocess_scaled = ColumnTransformer(
    transformers=[
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]), numeric_features)
    ],
    remainder="drop"
)

# 트리 계열은 스케일링 필수 아님
preprocess_noscale = ColumnTransformer(
    transformers=[
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
        ]), numeric_features)
    ],
    remainder="drop"
)

# ---------------------------------------------------------
# 3) 비교할 모델들 (원래 + XGB/Cat)
# ---------------------------------------------------------
try:
    from xgboost import XGBRegressor
except Exception as e:
    XGBRegressor = None
    print("⚠️ xgboost import 실패:", e)

try:
    from catboost import CatBoostRegressor
except Exception as e:
    CatBoostRegressor = None
    print("⚠️ catboost import 실패:", e)

models = {
    "ExtraTrees": Pipeline([
        ("prep", preprocess_noscale),
        ("model", ExtraTreesRegressor(
            n_estimators=800,
            random_state=42,
            n_jobs=-1
        ))
    ]),
    "RandomForest": Pipeline([
        ("prep", preprocess_noscale),
        ("model", RandomForestRegressor(
            n_estimators=800,
            random_state=42,
            n_jobs=-1
        ))
    ]),
    "HistGB": Pipeline([
        ("prep", preprocess_noscale),
        ("model", HistGradientBoostingRegressor(
            random_state=42
        ))
    ]),
    "Ridge": Pipeline([
        ("prep", preprocess_scaled),
        ("model", Ridge(alpha=3.0))
    ]),
    "ElasticNet": Pipeline([
        ("prep", preprocess_scaled),
        ("model", ElasticNet(random_state=42))
    ]),
    "SVR(RBF)": Pipeline([
        ("prep", preprocess_scaled),
        ("model", SVR(kernel="rbf"))
    ]),
}

if XGBRegressor is not None:
    models["XGBoost"] = Pipeline([
        ("prep", preprocess_noscale),
        ("model", XGBRegressor(
            n_estimators=1000,
            learning_rate=0.05,
            max_depth=6,
            subsample=0.9,
            colsample_bytree=0.9,
            reg_lambda=1.0,
            random_state=42,
            n_jobs=-1,
            objective="reg:squarederror"
        ))
    ])

if CatBoostRegressor is not None:
    models["CatBoost"] = Pipeline([
        ("prep", preprocess_noscale),
        ("model", CatBoostRegressor(
            iterations=3000,
            learning_rate=0.03,
            depth=6,
            loss_function="MAE",
            random_seed=42,
            verbose=False
        ))
    ])

# ---------------------------------------------------------
# 4) LOBO 평가 함수
# ---------------------------------------------------------
def lobo_evaluate(model, X, y, groups, logo):
    fold_rows = []
    for train_idx, test_idx in logo.split(X, y, groups=groups):
        X_tr, X_te = X.iloc[train_idx], X.iloc[test_idx]
        y_tr, y_te = y[train_idx], y[test_idx]
        batt = groups[test_idx][0]

        model.fit(X_tr, y_tr)
        pred = model.predict(X_te)

        mae = mean_absolute_error(y_te, pred)
        rmse = mean_squared_error(y_te, pred) ** 0.5
        r2 = r2_score(y_te, pred)

        fold_rows.append([batt, len(test_idx), mae, rmse, r2])

    fold_df = pd.DataFrame(fold_rows, columns=["test_battery", "n_test", "MAE", "RMSE", "R2"])
    summary = {
        "MAE_mean": fold_df["MAE"].mean(),
        "MAE_std": fold_df["MAE"].std(),
        "RMSE_mean": fold_df["RMSE"].mean(),
        "RMSE_std": fold_df["RMSE"].std(),
        "R2_mean": fold_df["R2"].mean(),
        "R2_std": fold_df["R2"].std(),
        "n_folds": len(fold_df),
        "n_samples": len(X),
        "n_batteries": len(np.unique(groups)),
    }
    return summary, fold_df

# ---------------------------------------------------------
# 5) 모델 전체 “딸깍 비교”
# ---------------------------------------------------------
all_summaries = []
all_folds = {}

for name, model in models.items():
    summary, fold_df = lobo_evaluate(model, X, y, groups, logo)
    all_summaries.append([
        name,
        summary["MAE_mean"], summary["RMSE_mean"], summary["R2_mean"],
        summary["MAE_std"], summary["RMSE_std"], summary["R2_std"],
        summary["n_folds"], summary["n_samples"], summary["n_batteries"]
    ])
    all_folds[name] = fold_df

result = pd.DataFrame(all_summaries, columns=[
    "model", "MAE", "RMSE", "R2",
    "MAE_std", "RMSE_std", "R2_std",
    "n_folds", "n_samples", "n_batteries"
]).sort_values("MAE")

print("=" * 90)
print(f"[LOBO AutoML + TS Features] Target = {TARGET} | Batteries = {sorted(df['battery_id'].unique())}")
print(f"Base feats: {len(BASE_FEATURES)} | +lags:{len(LAG_FEATURES)} +diff:{len(DIFF_FEATURES)} +roll:{len(ROLL_FEATURES)} => total {len(FEATURES)}")
print("=" * 90)
display(result)

# ---------------------------------------------------------
# 6) 베스트 모델 배터리별 성능
# ---------------------------------------------------------
best_name = result.iloc[0]["model"]
print("\nBest model:", best_name)
display(all_folds[best_name].sort_values("MAE"))

data shape (after ts features): (620, 23)


,battery_id,cycle_id,SOH,discharge_t_to_V_below_3.5,discharge_V_mean,slope_Vmeas_50_1500,HI2_Max_Temp,discharge_E_Wh_abs,discharge_t_to_V_below_3.5_lag1,discharge_V_mean_lag1,...,discharge_t_to_V_below_3.5_diff1,discharge_V_mean_diff1,slope_Vmeas_50_1500_diff1,HI2_Max_Temp_diff1,discharge_E_Wh_abs_diff1,discharge_t_to_V_below_3.5_rm5,discharge_V_mean_rm5,slope_Vmeas_50_1500_rm5,HI2_Max_Temp_rm5,discharge_E_Wh_abs_rm5
0,B0005,5.0,0.988235,2121.794596,3.544875,-0.000238,38.665393,6.561344,2117.469577,3.546006,...,4.325019,-0.001131,-5.075432e-07,-0.096912,-0.002195,2097.847822,3.541710,-0.000236,38.852415,6.580198
1,B0005,6.0,0.988782,2133.519013,3.544142,-0.000239,38.751695,6.566809,2121.794596,3.544875,...,11.724417,-0.000733,-5.372219e-07,0.086302,0.005465,2115.314294,3.544142,-0.000237,38.806317,6.570111
2,B0005,7.0,0.988504,2142.412264,3.543990,-0.000239,38.820701,6.566596,2133.519013,3.544142,...,8.893251,-0.000152,-5.629843e-07,0.069006,-0.000212,2124.808686,3.544994,-0.000238,38.763778,6.564423
3,B0005,8.0,0.983447,2149.201807,3.555909,-0.000242,38.517130,6.543075,2142.412264,3.543990,...,6.789543,0.011919,-2.264793e-06,-0.303571,-0.023521,2132.879451,3.546984,-0.000239,38.703445,6.560273
4,B0005,9.0,0.982917,2146.916797,3.554758,-0.000241,38.526268,6.537439,2149.201807,3.555909,...,-2.285010,-0.001151,7.762757e-07,0.009138,-0.005637,2138.768895,3.548735,-0.000240,38.656237,6.555053


[LOBO AutoML + TS Features] Target = SOH | Batteries = ['B0005', 'B0006', 'B0007', 'B0018']
Base feats: 5 | +lags:5 +diff:5 +roll:5 => total 20


,model,MAE,RMSE,R2,MAE_std,RMSE_std,R2_std,n_folds,n_samples,n_batteries
3,Ridge,0.015300,0.018587,0.954826,0.010708,0.012934,0.042126,4,620,4
7,CatBoost,0.021422,0.026393,0.910892,0.017596,0.020340,0.093345,4,620,4
2,HistGB,0.025810,0.029311,0.893844,0.019840,0.021574,0.108754,4,620,4
0,ExtraTrees,0.026351,0.029478,0.895441,0.018286,0.019871,0.097134,4,620,4
6,XGBoost,0.026659,0.029543,0.895465,0.018042,0.019862,0.097142,4,620,4
1,RandomForest,0.027873,0.030635,0.887187,0.019635,0.021206,0.109115,4,620,4
5,SVR(RBF),0.058954,0.072375,0.410666,0.035809,0.040838,0.453146,4,620,4
4,ElasticNet,0.093852,0.109809,-0.300881,0.027358,0.030353,0.299666,4,620,4



Best model: Ridge


,test_battery,n_test,MAE,RMSE,R2
2,B0007,164,0.003621,0.005312,0.995956
0,B0005,164,0.009373,0.011918,0.986048
3,B0018,128,0.021148,0.022142,0.923116
1,B0006,164,0.027058,0.034975,0.914182


In [5]:
# 1. 전처리가 완료된 데이터셋(df_ts)에서 피처와 타겟 분리
# FEATURES 리스트에 lag, diff 피처들이 모두 포함되어 있습니다.
X_final = df_ts[FEATURES]
y_final = df_ts['SOH']

# 2. 1등 모델 가져오기 (Ridge)
best_model_pipeline = models[best_name]

# 3. 전체 데이터로 최종 학습
print(f"🏆 1등 모델({best_name})으로 최종 학습 및 저장을 시작합니다...")
best_model_pipeline.fit(X_final, y_final)

# 4. 모델 파일로 저장 (휴머노이드를 빼고 직관적인 이름으로 수정)
import joblib
joblib.dump(best_model_pipeline, 'battery_model.pkl')
print("✅ 모델 저장 완료! 현재 폴더에 'battery_model.pkl' 파일이 생성되었습니다.")

🏆 1등 모델(Ridge)으로 최종 학습 및 저장을 시작합니다...
✅ 모델 저장 완료! 현재 폴더에 'battery_model.pkl' 파일이 생성되었습니다.
